# 06 — Stream Fraud Detection

Reads card transactions arriving as JSON files in `lakehouse/streaming/landing/card_txns/`, applies a **velocity rule**, and writes fraud alerts to a **Delta sink**.

## The velocity rule

> A card is suspect when it sees **3 or more transactions of more than ₹20,000 within a 1-minute tumbling window.**

## How the streaming query is wired

| Concern | Choice |
|---|---|
| Source | `readStream.format("json")` on the landing dir, with **explicit schema** (no inference on streams) |
| Event time | `transaction_at` |
| Watermark | 30 seconds — tolerates late-arriving txns |
| Aggregation | tumbling 1-minute window per `card_id`, count of `amount > 20000` |
| Trigger | `processingTime='5 seconds'` |
| Output mode | **`update`** — emit as soon as a window's count crosses the threshold (no need to wait for watermark to close the window) |
| Sink | Delta via `foreachBatch` + `MERGE INTO` — keeps the alert table idempotent across re-emits |
| Lifetime | `awaitTermination(timeout=300)` — auto-stops after 5 minutes |

## Why `update` mode + `MERGE`?

Append mode would only emit a window after the watermark passes its end. In a short-lived demo where the producer stops before the watermark advances 75+ seconds past the last burst, those windows would never fire. Update mode emits each time a window's aggregate changes — and `MERGE INTO` makes the Delta sink idempotent against the resulting re-emits.

## Setup

In [ ]:
import sys
from pathlib import Path
if (Path.cwd() / "conf").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd() / "project" / "conf").exists():
    PROJECT_ROOT = Path.cwd() / "project"
else:
    raise RuntimeError("Run from inside project/ or repo root")
sys.path.insert(0, str(PROJECT_ROOT))

import shutil
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                                DoubleType, TimestampType)
from conf.spark import (get_spark, STREAM_LANDING_DIR,
                         STREAM_OUTPUT_DIR, STREAM_CHECKPOINT_DIR)

SOURCE_DIR     = STREAM_LANDING_DIR / "card_txns"
ALERTS_DIR     = STREAM_OUTPUT_DIR
CHECKPOINT_DIR = STREAM_CHECKPOINT_DIR / "fraud_alerts"

# Reset previous run so the demo starts clean
shutil.rmtree(ALERTS_DIR,     ignore_errors=True)
shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

spark = get_spark("FraudDetectStream")
spark.sparkContext.setLogLevel("WARN")
print(f"watching: {SOURCE_DIR}")
print(f"alerts  : {ALERTS_DIR}")

## Define the stream input schema

Streaming sources require an explicit schema — Spark won't infer over an unbounded stream.

In [ ]:
stream_schema = StructType([
    StructField("transaction_id",    StringType(),    False),
    StructField("card_id",           StringType(),    False),
    StructField("customer_id",       StringType(),    False),
    StructField("merchant_name",     StringType()),
    StructField("merchant_category", StringType()),
    StructField("amount",            DoubleType(),    False),
    StructField("currency",          StringType()),
    StructField("transaction_at",    TimestampType(), False),
    StructField("status",            StringType()),
])

txns_stream = (spark.readStream
    .schema(stream_schema)
    .option("maxFilesPerTrigger", 5)  # bound work per micro-batch
    .json(str(SOURCE_DIR)))

print("isStreaming:", txns_stream.isStreaming)

## Apply the velocity rule

In [ ]:
alerts = (txns_stream
    .filter(F.col("amount") > 20000)
    .withWatermark("transaction_at", "30 seconds")
    .groupBy(F.window("transaction_at", "1 minute").alias("window"),
             F.col("card_id"))
    .agg(F.count("*").alias("big_txn_count"),
         F.round(F.sum("amount"), 2).alias("big_txn_total"),
         F.first("customer_id").alias("customer_id"))
    .filter(F.col("big_txn_count") >= 3)
    .select(F.col("window.start").alias("window_start"),
            F.col("window.end").alias("window_end"),
            "card_id", "customer_id",
            "big_txn_count", "big_txn_total"))

alerts.printSchema()

## `foreachBatch` — idempotent MERGE into the Delta alert table

Each micro-batch may re-emit a window with an updated count (e.g. count went from 3 → 4). The MERGE keeps the alert table at one row per `(window_start, card_id)` and updates it in place.

In [ ]:
from delta.tables import DeltaTable

def upsert_alerts(batch_df, batch_id):
    if batch_df.rdd.isEmpty():
        return
    target_path = str(ALERTS_DIR)
    if not (ALERTS_DIR / "_delta_log").exists():
        # First non-empty batch creates the table
        batch_df.write.format("delta").mode("overwrite").save(target_path)
    else:
        target = DeltaTable.forPath(spark, target_path)
        (target.alias("t")
             .merge(batch_df.alias("s"),
                    "t.window_start = s.window_start AND t.card_id = s.card_id")
             .whenMatchedUpdateAll()
             .whenNotMatchedInsertAll()
             .execute())
    print(f"  batch {batch_id}: merged {batch_df.count()} alerts")

## Start the streaming queries and wait

Two queries side-by-side: one writes alerts to **Delta** via `foreachBatch`, the other prints them to the **console** so you can watch fraud bursts arrive in real time.

In [ ]:
delta_query = (alerts.writeStream
    .outputMode("update")
    .option("checkpointLocation", str(CHECKPOINT_DIR))
    .trigger(processingTime="5 seconds")
    .foreachBatch(upsert_alerts)
    .start())

console_query = (alerts.writeStream
    .format("console")
    .option("truncate", False)
    .outputMode("update")
    .trigger(processingTime="5 seconds")
    .start())

print("Both streaming queries started. Will run for ~5 minutes.")
print("In another kernel, run 05-stream-producer.ipynb to feed transactions in.")

TIMEOUT_SEC = 300
delta_query.awaitTermination(timeout=TIMEOUT_SEC)
delta_query.stop()
console_query.stop()
print("Streaming queries stopped.")

## Inspect the durable fraud-alerts Delta table

In [ ]:
if (ALERTS_DIR / "_delta_log").exists():
    df_alerts = spark.read.format("delta").load(str(ALERTS_DIR))
    print(f"alerts written: {df_alerts.count()}")
    df_alerts.orderBy("window_start").show(50, truncate=False)
else:
    print("No alerts written. Did the producer (notebook 05) run while this notebook was streaming?")

## Summary

End-to-end streaming fraud detection: file source → watermarked windowed aggregation → Delta sink. The fraud_alerts Delta table is queryable just like any other gold table.